# MapBiomas Fire Catalog - Venezuela - COLLECTION_02

**Campaign:** `venezuela.collection_02` | **Target:** `burned_area` | **Sensor:** landsat | **Periodicity:** yearly

Thin entry point. The pipeline code lives in `network/catalog/development/version_04/src/core`.


# MapBiomas Fire — Pipeline (M0-M9)

## Documentation and User Guide
Short guide to the pipeline. Run the sections in order: each module reads what the previous one saved.

#### How it works (5 steps)

1. **Images (M1-M2):** GEE builds one mosaic per period (`minnbr`: the lowest-NBR observation, where a burn scar shows best) and exports it to GCS. M2 joins the pieces into Cloud-Optimized GeoTIFFs (COGs).
2. **Samples (M3):** in the GEE toolkit, draw `fire` and `notFire` polygons for each region and period.
3. **Training (M4):** a neural network (DNN) learns burned vs. unburned from the samples. Default: 5 hidden layers `7, 14, 7, 14, 7`, no dropout, 7000 iterations (same setup as the legacy collection).
4. **Classification (M5):** the model scans the region tile by tile and saves the burn probability and the day of the burn for every pixel.
5. **Publication and post-processing (M6 -> M7-M9):** M6 builds regional mosaics and statistics and uploads them to GEE. M7-M9 run in GEE: national merge, filters, statistics and promotion.

#### Modules

| Module | Runs in | What it does | Output |
|---|---|---|---|
| **M0** | Colab | Setup, paths (`set_global_opts()`), authentication | config printed as `[CONFIG]`, `[GEE]`, `[GCS]` |
| **M1** | Colab -> GEE | Exports period mosaics | GCS `CHUNKS/` |
| **M2** | Colab | Assembles chunks into COGs | GCS `COG/` |
| **M3** | GEE toolkit | Training polygons | GEE + GCS `LIBRARY_SAMPLES/` |
| **M4** | Colab | Trains the DNN | GCS `LIBRARY_MODELS/{model}/` |
| **M5** | Colab | Classifies tiles | GCS `CLASSIFIED_TILES/` |
| **M6** | Colab | Regional mosaics, stats, GEE upload | GCS `CLASSIFIED_REGION/`, `STATS/`; GEE `REGIONAL/` |
| **M7** | GEE | ft00 national merge, ft01 land-use + small patches, ft02 temporal filter | GEE `FILTERED/` |
| **M8** | GEE | Burned-area statistics per filter stage | tables |
| **M9** | GEE | Promotion of the chosen version | pre-official collection |

#### Classification output (M5)

Each tile is an int16 GeoTIFF with 2 bands (nodata `-9999`):

| Band | Name | Value |
|---|---|---|
| b1 | `probability` | burn probability x 1000 (0-1000; `50` = 5%), every valid pixel |
| b2 | `dayOfYear` | day of the year of the burn, where `probability > 0` |

Because the probability is kept, a stricter burned cut can be tested later in GEE without classifying again.

#### Where things are saved

```
GCS {gcs_library_images_prefix}/LIBRARY_IMAGES/{SENSOR}/{PERIODICITY}/{MOSAIC}/{period}/
    CHUNKS/  (M1)    COG/  (M2)

GCS {gcs_campaigns_prefix}/{campaign}/
    LIBRARY_SAMPLES/                    M3  sample_{id}_{region}_{period}.csv
    LIBRARY_MODELS/{model}/             M4  weights, metadata, metrics
    LIBRARY_CLASSIFICATIONS/{model}/
        CLASSIFIED_TILES/               M5  tile_{region}_{cell}_{period}.tif
        CLASSIFIED_REGION/              M6  region_{region}_{model}_{period}.tif
        STATS/                          M6  stats_tile.csv, stats_region.csv

GEE {gee_campaigns_prefix}/{campaign}/
    AUXILIARY/                          M0  fire regions
    LIBRARY_SAMPLES/                    M3  sample polygons
    LIBRARY_CLASSIFICATIONS/REGIONAL/   M6  probability + dayOfYear per region
    LIBRARY_CLASSIFICATIONS/FILTERED/   M7  ft00, ft01, ft02
```

Model name: `training_{id}_{shortname}_{sensor}` (e.g. `training_0001_amazon_sentinel2`).

#### Good to know

- Colab and local disk are temporary. Everything that matters is saved in GCS and GEE.
- Re-running is safe: modules skip what already exists in GCS. To classify again with a new model, train it under a new name (or delete the old tiles).
- In the selection lists, **Select remaining** marks only pending items; re-marking a green (done) item overwrites it.
- Access: GEE uses your own account; GCS uses the service account. If M0 prints `[GCS][WARN]`, ask to join the service-account group.

## [M0] — Environment Setup
Clones the repository (first run only), installs system and Python dependencies (GDAL, gcsfs, rasterio, earthengine-api), configures GCS bucket and GEE project via `set_global_opts()`, authenticates with Earth Engine, and registers the region FeatureCollection.

In [ ]:
# M0 -- Colab Setup -- Venezuela | landsat | yearly | COLLECTION_02
import sys, os, subprocess, shutil

# -- Always create a fresh clone (safe to rerun from any working directory) --
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Old clone removed.")
subprocess.run(["git", "clone", "--branch", "main", "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)

# -- Install system dependencies --
subprocess.run(["apt-get", "update", "-qq"])
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"])
subprocess.run(["pip", "install", "-q", "earthengine-api", "gcsfs", "rasterio", "scipy", "tqdm"])

# -- Add to Python path --
repo_path = os.path.join(REPO_DIR, "network/catalog/development/version_04/src/core")
sys.path = [p for p in sys.path if os.path.abspath(p or os.curdir) != repo_path]
sys.path.insert(0, repo_path)
os.chdir(repo_path)

# -- Configure project --
from M0_auth_config import set_global_opts, authenticate, print_config

COUNTRY_SELECTED = 'venezuela'                          # must match an existing catalog
CAMPAIGN_SELECTED = 'COLLECTION_02'                     # must match an existing campaign folder
# Naming convention:
#   country      -> lowercase country code/name (e.g. brazil, bolivia, peru)
#   campaign     -> uppercase; MONITOR_01 for monthly monitoring,
#                   COLLECTION_02/COLLECTION_06 for annual collections
#   task flag    -> MONITOR for monthly monitor campaigns, FUEGO for annual collections
#   region asset -> CATALOG_01/{CAMPAIGN}/AUXILIARY/regiones_fuego_{country}_v1
#   New region/sensor/period: create a new campaign folder under CATALOG_01/{CAMPAIGN}
#   and keep the same GCS/GEE roots and region asset pattern used below.
# Enabled now: sensor sentinel2|landsat|hls|modis; periodicity monthly|yearly.
# Reserved for later: viirs; semestral/quarterly/bimonthly/biennial.
set_global_opts(
    # === Active Campaign ===
    country=COUNTRY_SELECTED,
    campaign=CAMPAIGN_SELECTED,

    # === Pipeline Settings ===
    sensor='landsat',                                   # active: sentinel2 | landsat | hls | modis (viirs later)
    periodicity='yearly',                               # active: monthly | yearly (one only; others disabled)
    mosaic_methods='minnbr',                            # active: minnbr | minnbr_buffer | median | minndvi
    personal_task_flag='FUEGO',                         # free prefix for GEE task names
    language='es',                                      # en | es | pt | fr | id

    # === Google Cloud Storage ===
    gcs_bucket='mapbiomas-fire',
    gcs_library_images_prefix=f'sudamerica/{COUNTRY_SELECTED}/CATALOG_01',
    gcs_campaigns_prefix=f'sudamerica/{COUNTRY_SELECTED}/CATALOG_01',

    # === Google Earth Engine ===
    gee_project=f'mapbiomas-{COUNTRY_SELECTED}',
    gee_library_images_prefix=f'projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01',
    gee_campaigns_prefix=f'projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01',
    # Region FeatureCollection for this campaign (one per country/campaign)
    asset_regions=f'projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01/{CAMPAIGN_SELECTED}/AUXILIARY/regiones_fuego_{COUNTRY_SELECTED}_v1',
)

authenticate()
print_config()


## [M1] — Export (GEE → GCS)
Multi-sensor satellite image export from GEE ImageCollections to GCS. Handles Sentinel-2, Landsat (5/7/8/9), HLS and MODIS with sensor-specific radiometric corrections, cloud masking, and configurable mosaic types (MINNBR, MINNBR_BUFFER, etc.).

In [ ]:
from M1_export_ui import run_ui, start_export

ui_exporter = run_ui(years=[])

In [ ]:
# Export trigger (GEE -> GCS)

start_export(ui_exporter)

## [M2] — Mosaic Assembly (COG)

Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT virtual stacking. Produces compressed, tiled COGs with DEFLATE compression. 
Check exported assets and errors using this notebook validation tool:
https://colab.research.google.com/github/mapbiomas/brazil-fire/blob/main/network/catalog/production/notebooks/validate_gcs_export.ipynb

In [ ]:
from M2_mosaic_ui import run_ui, start_mosaic_assembly
ui_assembler = run_ui(years=[])

In [ ]:
# Mosaic trigger (Local/GCS COG)
start_mosaic_assembly(ui_assembler)

## [M3] — Sample Collection (GEE Toolkit)
Training data collection via a custom GEE JavaScript Toolkit. Users draw fire/notFire polygons on satellite imagery. Samples are exported to GEE Assets and GCS with full metadata (satellite, date, region, campaign).

In [ ]:
from M3_sample_ui import show_toolkit_links
show_toolkit_links()

## [M4] — DNN Training
Trains a Deep Neural Network classifier using a flexible band extraction matrix. Samples pixels from M2 mosaics using M3 polygons, trains a configurable DNN, generates t-SNE audit plots, and saves model weights and metadata to GCS.

In [ ]:
from M4_model_trainer import run_ui, start_training
ui_trainer = run_ui()

In [ ]:
# Training trigger
start_training(ui_trainer)

## [M5] — Tile-Based Classification
Tile-based burned area classification using a trained DNN model. Loads the model from GCS, retrieves the cell grid for the target region, classifies each tile independently, and generates per-tile classified rasters with local statistics. Results are consumed by M6 for mosaicking and publication.

In [ ]:
# 1. M5 Scheduling Interface
from M5_classifier_ui import run_m5_ui
ui_m5 = run_m5_ui(years=[], periodicity_active=["yearly"])

In [ ]:
# 2. M5 Async Processing Engine
# n_workers: auto (cpu_count). Force com: run_m5_workplan(n_workers=4, progress_callback=ui_m5._on_tile_progress)
from M5_classifier import run_m5_workplan
run_m5_workplan(progress_callback=ui_m5._on_tile_progress)

## [M6] — Mosaic, Statistics & Publication
Assembles classified tiles into regional mosaics, computes burned area statistics, and publishes results to Google Earth Engine as ImageCollections. Provides an interactive dashboard for monitoring batch progress, analytics, and coverage.

In [ ]:
# 3. M6 Scheduling Interface
from M6_ui import run_m6_ui
ui_m6 = run_m6_ui()

In [ ]:
# 2. M6 Publish (mosaic + stats + GEE upload)
# Runs only when classified tiles (COMPLETED) are available in GCS.
# upload_gee=True sends regional mosaics to Google Earth Engine.
from M6_publisher import run_m6_publish
run_m6_publish(upload_gee=True)

## [M7-M9] — Post-processing in Google Earth Engine
These steps are ready and run in the **GEE Code Editor**, not in this notebook. They read the regional images that M6 uploaded to GEE.

| Step | What it does | GEE script |
|---|---|---|
| **M7 ft00** | Picks one model per region and merges them into a national image | [ft00_selection_merge](https://code.earthengine.google.com/?scriptPath=users/mapbiomasworkspace1/mapbiomas-fire:5-MapBiomas-Fuego/M7/ft00_selection_merge) |
| **M7 ft01** | Land-use mask + removal of small isolated patches | [ft01_lulc_filters](https://code.earthengine.google.com/?scriptPath=users/mapbiomasworkspace1/mapbiomas-fire:5-MapBiomas-Fuego/M7/venezuela/COLLECTION_02/yearly/ft01_lulc_filters) |
| **M7 ft02** | Temporal filter (consistency with previous periods) | [ft02_temporal_filter](https://code.earthengine.google.com/?scriptPath=users/mapbiomasworkspace1/mapbiomas-fire:5-MapBiomas-Fuego/M7/venezuela/COLLECTION_02/yearly/ft02_temporal_filter) |
| **M8** | Burned-area statistics per filter stage and land use | [M8_statistics](https://code.earthengine.google.com/?scriptPath=users/mapbiomasworkspace1/mapbiomas-fire:5-MapBiomas-Fuego/M8/M8_statistics) |
| **M9** | Promotes the chosen version to pre-official | [M9_00_promote_ui](https://code.earthengine.google.com/?scriptPath=users/mapbiomasworkspace1/mapbiomas-fire:5-MapBiomas-Fuego/M9/M9_00_promote_ui) |

The links need access to the `mapbiomas-fire` GEE repository. Copies of the scripts are kept in `src/gee/` of this version.